In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

budget = pd.read_csv(
    "../data/Budget_Clean.csv",
    sep=";",
    encoding="latin1"
)

pnl = pd.read_csv(
    "../data/Pnl_Clean.csv",
    sep=";",
    encoding="latin1"
)

budget = budget.loc[:, ~budget.columns.str.contains("^Unnamed")]
pnl = pnl.loc[:, ~pnl.columns.str.contains("^Unnamed")]

budget["Month_Date"] = pd.to_datetime(
    budget["Month_Date"],
    dayfirst=True
)

pnl["Month_Date"] = pd.to_datetime(
    pnl["Month_Date"],
    dayfirst=True
)

print(budget.dtypes)
print(pnl.dtypes)


Category                object
Account Name            object
Year                     int64
Month_Date      datetime64[ns]
Month_Label             object
Rate                   float64
Budget_COP             float64
Budget_GBP             float64
dtype: object
Month_Number             int64
Month_Date      datetime64[ns]
Month_Label             object
Account                  int64
Description             object
Code                    object
Actual_GBP             float64
Actual_COP             float64
Rate                   float64
dtype: object


Inspeccionar Columnas

In [2]:
print("BUDGET COLUMNS")
print(budget.columns)

print("\nPNL COLUMNS")
print(pnl.columns)

BUDGET COLUMNS
Index(['Category', 'Account Name', 'Year', 'Month_Date', 'Month_Label', 'Rate',
       'Budget_COP', 'Budget_GBP'],
      dtype='object')

PNL COLUMNS
Index(['Month_Number', 'Month_Date', 'Month_Label', 'Account', 'Description',
       'Code', 'Actual_GBP', 'Actual_COP', 'Rate'],
      dtype='object')


Limpieza de columnas "Unnamed"

In [3]:
#Verificar Data Types
print(budget.dtypes)

print("\n")

print(pnl.dtypes)

Category                object
Account Name            object
Year                     int64
Month_Date      datetime64[ns]
Month_Label             object
Rate                   float64
Budget_COP             float64
Budget_GBP             float64
dtype: object


Month_Number             int64
Month_Date      datetime64[ns]
Month_Label             object
Account                  int64
Description             object
Code                    object
Actual_GBP             float64
Actual_COP             float64
Rate                   float64
dtype: object


Fact Table Creation for Budget and Actuals

In [4]:
budget_fact = budget[[
    "Month_Date",
    "Month_Label",
    "Category",
    "Account Name",
    "Budget_COP",
    "Budget_GBP"
]].copy()

budget_fact["Scenario"] = "Budget"

budget_fact.rename(columns={
    "Account Name": "Account",
    "Budget_COP": "Amount_COP",
    "Budget_GBP": "Amount_GBP"
}, inplace=True)

In [5]:
pnl_fact = pnl[[
    "Month_Date",
    "Month_Label",
    "Description",
    "Actual_COP",
    "Actual_GBP"
]].copy()

pnl_fact["Scenario"] = "Actual"

pnl_fact.rename(columns={
    "Description": "Account",
    "Actual_COP": "Amount_COP",
    "Actual_GBP": "Amount_GBP"
}, inplace=True)

pnl_fact["Category"] = "Actuals"

In [6]:
finance = pd.concat([budget_fact, pnl_fact], ignore_index=True)

In [7]:
monthly = finance.groupby(
    ["Month_Date", "Scenario"],
    as_index=False
)["Amount_COP"].sum()

monthly = monthly.sort_values("Month_Date")

monthly

,Month_Date,Scenario,Amount_COP
0,2025-07-01,Actual,-5.592165e+08
1,2025-07-01,Budget,-1.260433e+09
2,2025-08-01,Actual,-3.601962e+08
3,2025-08-01,Budget,-1.278128e+09
4,2025-09-01,Actual,-3.667514e+08
5,2025-09-01,Budget,-1.341478e+09
6,2025-10-01,Actual,-7.600680e+08
7,2025-10-01,Budget,-1.277287e+09
8,2025-11-01,Actual,-4.019191e+08
9,2025-11-01,Budget,-1.302543e+09


Visualización

In [14]:
monthly = finance.groupby(
    ["Month_Date", "Scenario"],
    as_index=False
)["Amount_COP"].sum()

monthly = monthly.sort_values("Month_Date")

fig = px.line(
    monthly,
    x="Month_Date",
    y="Amount_COP",
    color="Scenario",
    markers=True,
    title="FY26 Monthly Financial Performance"
)

fig.update_traces(
    line=dict(width=3),
    marker=dict(size=8)
)

fig.update_layout(
    template="plotly_dark",
    height=650,

    title=dict(
        text="FY26 Monthly Financial Performance",
        x=0.02,
        xanchor="left",
        font=dict(size=26)
    ),

    plot_bgcolor="#111111",
    paper_bgcolor="#111111",

    font=dict(
        family="Arial",
        size=14
    ),

    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),

    hovermode="x unified",

    margin=dict(
        l=40,
        r=40,
        t=80,
        b=40
    ),

    xaxis_title="",
    yaxis_title="COP Amount"
)

fig.update_xaxes(
    tickformat="%b-%y",
    dtick="

SyntaxError: unterminated string literal (detected at line 64) (3426486210.py, line 64)

KPIs de Tarjetas Ejecutivas

In [9]:
total_budget = finance.loc[
    finance["Scenario"] == "Budget",
    "Amount_COP"
].sum()

total_actual = finance.loc[
    finance["Scenario"] == "Actual",
    "Amount_COP"
].sum()

variance = total_actual - total_budget

execution_pct = (total_actual / total_budget) * 100

print(f"Total Budget: {total_budget:,.0f}")
print(f"Total Actual: {total_actual:,.0f}")
print(f"Variance: {variance:,.0f}")
print(f"Execution %: {execution_pct:.1f}%")

Total Budget: -14,776,798,855
Total Actual: -5,654,588,606
Variance: 9,122,210,249
Execution %: 38.3%


In [10]:
fig = go.Figure()

fig.add_trace(go.Indicator(
    mode="number",
    value=abs(total_budget),
    title={"text": "Total Budget (COP)"},
    number={"prefix": "$", "valueformat": ",.0f"},
    domain={"x": [0, 0.24], "y": [0, 1]}
))

fig.add_trace(go.Indicator(
    mode="number",
    value=abs(total_actual),
    title={"text": "Total Actual (COP)"},
    number={"prefix": "$", "valueformat": ",.0f"},
    domain={"x": [0.26, 0.5], "y": [0, 1]}
))

fig.add_trace(go.Indicator(
    mode="number",
    value=abs(variance),
    title={"text": "Variance (COP)"},
    number={"prefix": "$", "valueformat": ",.0f"},
    domain={"x": [0.52, 0.76], "y": [0, 1]}
))

fig.add_trace(go.Indicator(
    mode="gauge+number",
    value=abs(execution_pct),
    title={"text": "Execution %"},
    number={"suffix": "%"},
    gauge={
        "axis": {"range": [0, 100]},
        "bar": {"color": "lightgreen"}
    },
    domain={"x": [0.78, 1], "y": [0, 1]}
))

fig.update_layout(
    template="plotly_dark",
    height=300,
    margin=dict(t=40, b=20, l=20, r=20)
)

fig.show()

In [11]:
monthly_cop = finance.groupby(
    ["Month_Date", "Scenario"],
    as_index=False
)["Amount_COP"].sum()

monthly_gbp = finance.groupby(
    ["Month_Date", "Scenario"],
    as_index=False
)["Amount_GBP"].sum()

monthly_cop = monthly_cop.sort_values("Month_Date")
monthly_gbp = monthly_gbp.sort_values("Month_Date")

In [17]:
monthly = finance.groupby(
    ["Month_Date", "Scenario"],
    as_index=False
)["Amount_COP"].sum()

monthly = monthly.sort_values("Month_Date")

fig = px.line(
    monthly,
    x="Month_Date",
    y="Amount_COP",
    color="Scenario",
    markers=True,
    title="FY26 Monthly Financial Performance"
)

fig.update_traces(
    line=dict(width=3),
    marker=dict(size=8)
)

fig.update_layout(
    template="plotly_dark",
    height=650,

    title=dict(
        text="FY26 Monthly Financial Performance",
        x=0.02,
        xanchor="left",
        font=dict(size=26)
    ),

    plot_bgcolor="#111111",
    paper_bgcolor="#111111",

    font=dict(
        family="Arial",
        size=14
    ),

    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),

    hovermode="x unified",

    margin=dict(
        l=40,
        r=40,
        t=80,
        b=40
    ),

    xaxis_title="",
    yaxis_title="COP Amount"
)

fig.update_xaxes(
    tickformat="%b-%y",
    dtick="M1",
    showgrid=False
)

fig.update_yaxes(
    tickformat=",.2s",
    gridcolor="rgba(255,255,255,0.08)"
)

fig.show()

## Contruccion de MTD/YTD

In [18]:
latest_actual_month = finance.loc[
    finance["Scenario"] == "Actual",
    "Month_Date"
].max()

print(latest_actual_month)

2026-04-01 00:00:00


In [19]:
mtd_data = finance[
    finance["Month_Date"] == latest_actual_month
]

ytd_data = finance[
    finance["Month_Date"] <= latest_actual_month
]

In [20]:
report_cop = pd.DataFrame({
    "Metric": ["Budget", "Actual", "Variance", "Execution %"],

    "MTD": [
        mtd_data.loc[mtd_data["Scenario"] == "Budget", "Amount_COP"].sum(),
        mtd_data.loc[mtd_data["Scenario"] == "Actual", "Amount_COP"].sum(),
        (
            mtd_data.loc[mtd_data["Scenario"] == "Actual", "Amount_COP"].sum()
            -
            mtd_data.loc[mtd_data["Scenario"] == "Budget", "Amount_COP"].sum()
        ),
        (
            mtd_data.loc[mtd_data["Scenario"] == "Actual", "Amount_COP"].sum()
            /
            mtd_data.loc[mtd_data["Scenario"] == "Budget", "Amount_COP"].sum()
        ) * 100
    ],

    "YTD": [
        ytd_data.loc[ytd_data["Scenario"] == "Budget", "Amount_COP"].sum(),
        ytd_data.loc[ytd_data["Scenario"] == "Actual", "Amount_COP"].sum(),
        (
            ytd_data.loc[ytd_data["Scenario"] == "Actual", "Amount_COP"].sum()
            -
            ytd_data.loc[ytd_data["Scenario"] == "Budget", "Amount_COP"].sum()
        ),
        (
            ytd_data.loc[ytd_data["Scenario"] == "Actual", "Amount_COP"].sum()
            /
            ytd_data.loc[ytd_data["Scenario"] == "Budget", "Amount_COP"].sum()
        ) * 100
    ]
})

report_cop

,Metric,MTD,YTD
0,Budget,-1.210598e+09,-1.240867e+10
1,Actual,-3.636416e+08,-5.654589e+09
2,Variance,8.469563e+08,6.754083e+09
3,Execution %,3.003818e+01,4.556965e+01


In [21]:
report_display = report_cop.copy()

report_display.loc[
    report_display["Metric"] != "Execution %",
    ["MTD", "YTD"]
] = report_display.loc[
    report_display["Metric"] != "Execution %",
    ["MTD", "YTD"]
].applymap(lambda x: f"${x/1e9:.1f}B")

report_display.loc[
    report_display["Metric"] == "Execution %",
    ["MTD", "YTD"]
] = report_display.loc[
    report_display["Metric"] == "Execution %",
    ["MTD", "YTD"]
].applymap(lambda x: f"{x:.1f}%")

report_display

C:\Users\HP\AppData\Local\Temp\ipykernel_20060\2873046374.py:9: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  ].applymap(lambda x: f"${x/1e9:.1f}B")
C:\Users\HP\AppData\Local\Temp\ipykernel_20060\2873046374.py:3: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['$-1.2B' '$-0.4B' '$0.8B']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  report_display.loc[
C:\Users\HP\AppData\Local\Temp\ipykernel_20060\2873046374.py:3: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['$-12.4B' '$-5.7B' '$6.8B']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  report_display.loc[
C:\Users\HP\AppData\Local\Temp\ipykernel_20060\2873046374.py:17: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.

,Metric,MTD,YTD
0,Budget,$-1.2B,$-12.4B
1,Actual,$-0.4B,$-5.7B
2,Variance,$0.8B,$6.8B
3,Execution %,30.0%,45.6%


In [23]:
ytd_budget = ytd_data.loc[
    ytd_data["Scenario"] == "Budget",
    "Amount_COP"
].sum()

ytd_actual = ytd_data.loc[
    ytd_data["Scenario"] == "Actual",
    "Amount_COP"
].sum()

ytd_variance = ytd_actual - ytd_budget

ytd_execution = (ytd_actual / ytd_budget) * 100

In [24]:
fig = go.Figure()

fig.add_trace(go.Indicator(
    mode="number",
    value=abs(ytd_budget),
    title={"text": "YTD Budget"},
    number={"prefix": "$", "valueformat": ".2s"},
    domain={"x": [0.00, 0.24], "y": [0, 1]}
))

fig.add_trace(go.Indicator(
    mode="number",
    value=abs(ytd_actual),
    title={"text": "YTD Actual"},
    number={"prefix": "$", "valueformat": ".2s"},
    domain={"x": [0.26, 0.50], "y": [0, 1]}
))

fig.add_trace(go.Indicator(
    mode="number",
    value=abs(ytd_variance),
    title={"text": "Variance"},
    number={"prefix": "$", "valueformat": ".2s"},
    domain={"x": [0.52, 0.76], "y": [0, 1]}
))

fig.add_trace(go.Indicator(
    mode="number",
    value=abs(ytd_execution),
    title={"text": "Execution %"},
    number={"suffix": "%", "valueformat": ".1f"},
    domain={"x": [0.78, 1], "y": [0, 1]}
))

fig.update_layout(
    template="plotly_dark",
    height=250,
    margin=dict(t=40, b=20, l=20, r=20),
    title={
        "text": "FY26 YTD Financial Summary",
        "x": 0.02,
        "font": {"size": 24}
    }
)

fig.show()

In [25]:
mtd_budget = mtd_data.loc[
    mtd_data["Scenario"] == "Budget",
    "Amount_COP"
].sum()

mtd_actual = mtd_data.loc[
    mtd_data["Scenario"] == "Actual",
    "Amount_COP"
].sum()

mtd_variance = mtd_actual - mtd_budget

mtd_execution = (mtd_actual / mtd_budget) * 100

In [26]:
fig = go.Figure()

# YTD KPIs
fig.add_trace(go.Indicator(
    mode="number",
    value=abs(ytd_budget),
    title={"text": "Budget"},
    number={"prefix": "$", "valueformat": ".2s"},
    domain={"x": [0.00, 0.24], "y": [0, 1]},
    visible=True
))

fig.add_trace(go.Indicator(
    mode="number",
    value=abs(ytd_actual),
    title={"text": "Actual"},
    number={"prefix": "$", "valueformat": ".2s"},
    domain={"x": [0.26, 0.50], "y": [0, 1]},
    visible=True
))

fig.add_trace(go.Indicator(
    mode="number",
    value=abs(ytd_variance),
    title={"text": "Variance"},
    number={"prefix": "$", "valueformat": ".2s"},
    domain={"x": [0.52, 0.76], "y": [0, 1]},
    visible=True
))

fig.add_trace(go.Indicator(
    mode="number",
    value=abs(ytd_execution),
    title={"text": "Execution %"},
    number={"suffix": "%", "valueformat": ".1f"},
    domain={"x": [0.78, 1], "y": [0, 1]},
    visible=True
))

# MTD KPIs
fig.add_trace(go.Indicator(
    mode="number",
    value=abs(mtd_budget),
    title={"text": "Budget"},
    number={"prefix": "$", "valueformat": ".2s"},
    domain={"x": [0.00, 0.24], "y": [0, 1]},
    visible=False
))

fig.add_trace(go.Indicator(
    mode="number",
    value=abs(mtd_actual),
    title={"text": "Actual"},
    number={"prefix": "$", "valueformat": ".2s"},
    domain={"x": [0.26, 0.50], "y": [0, 1]},
    visible=False
))

fig.add_trace(go.Indicator(
    mode="number",
    value=abs(mtd_variance),
    title={"text": "Variance"},
    number={"prefix": "$", "valueformat": ".2s"},
    domain={"x": [0.52, 0.76], "y": [0, 1]},
    visible=False
))

fig.add_trace(go.Indicator(
    mode="number",
    value=abs(mtd_execution),
    title={"text": "Execution %"},
    number={"suffix": "%", "valueformat": ".1f"},
    domain={"x": [0.78, 1], "y": [0, 1]},
    visible=False
))

fig.update_layout(
    template="plotly_dark",
    height=250,

    title={
        "text": "Financial Performance Summary",
        "x": 0.02,
        "font": {"size": 24}
    },

    margin=dict(
        t=80,
        b=20,
        l=20,
        r=20
    ),

    updatemenus=[
        dict(
            type="buttons",
            direction="right",
            buttons=[
                dict(
                    label="YTD",
                    method="update",
                    args=[
                        {"visible": [True, True, True, True, False, False, False, False]},
                        {"title": "Financial Performance Summary — YTD"}
                    ]
                ),
                dict(
                    label="MTD",
                    method="update",
                    args=[
                        {"visible": [False, False, False, False, True, True, True, True]},
                        {"title": "Financial Performance Summary — MTD"}
                    ]
                )
            ],
            x=0.5,
            xanchor="center",
            y=1.20,
            yanchor="top"
        )
    ]
)

fig.show()